In [ ]:
%%capture
import os
os.environ["CUDA_VISIBLE_DEVICES"] = "0"

!pip install pip3-autoremove
!pip install torch torchvision torchaudio xformers --index-url https://download.pytorch.org/whl/cu124
!pip install unsloth==2025.9.4
!pip install --upgrade transformers==4.56.1 "huggingface_hub>=0.34.0" "datasets>=3.4.1,<4.0.0"

In [ ]:
import ast
import torch
import random
import numpy as np
import pandas as pd
from tqdm import tqdm
from huggingface_hub import login
from unsloth import FastLanguageModel
from datasets import Dataset, load_dataset
from unsloth.chat_templates import train_on_responses_only
from transformers import AutoTokenizer, StoppingCriteria, StoppingCriteriaList

login('your_huggingface_auth_token_here')

In [ ]:
model, _ = FastLanguageModel.from_pretrained(
    model_name="./models/ViLegalQwen3-1.7B-Base",
    max_seq_length=4096,
    dtype=torch.float16,
    load_in_4bit=True,
    token = "your_huggingface_auth_token_here",
)

tokenizer = AutoTokenizer.from_pretrained("Qwen/Qwen3-1.7B-Base")

In [ ]:
model = FastLanguageModel.get_peft_model(
    model,
    r=16,
    lora_alpha=32,
    lora_dropout=0.05,
    bias="none",
    use_gradient_checkpointing="unsloth",
    random_state=42,
    use_rslora=False,  # We support rank stabilized LoRA
    loftq_config=None, # And LoftQ
    target_modules=["q_proj", "k_proj", "v_proj", "o_proj",
                      "gate_proj", "up_proj", "down_proj",],
    modules_to_save=['embed_tokens', 'lm_head']
)

In [ ]:
# We used this dataset from BOSCH@AI_Team (https://huggingface.co/datasets/QuangTran276/new_reasoning), but for convenient, we had re-preprocessed this dataset into a standard format.

df = df = pd.read_csv("./datasets/processed/new_reasoning/syllogism-reasoning.csv")
df.columns = ["question", "thinking_content", "answer"]
df_val = df.sample(144, random_state=42)
df_train = df.drop(index=df_val.index)
df_train = df_train.reset_index(drop=True)
df_val = df_val.reset_index(drop=True)
df_test = pd.read_parquet("hf://datasets/VLSP2025-LegalSML/Public-Test/syllogism_questions/train-00000-of-00001.parquet")
len(df_train), len(df_val), len(df_test)

In [ ]:
df_test.to_csv("test_syllo.csv", index=False, encoding="utf-8-sig")

In [ ]:
SYSTEM_PROMPT = "Bạn là chuyên gia pháp luật Việt Nam. Phân tích và trả lời câu hỏi pháp lý theo cấu trúc tam đoạn luận:\n\nTiền đề lớn: Nêu quy định pháp luật áp dụng (Luật, Nghị định, Thông tư, Điều, Khoản)\nTiền đề nhỏ: Phân tích tình huống cụ thể và các yếu tố thực tế\nKết luận: Suy luận hậu quả pháp lý dựa trên việc áp dụng quy định vào tình huống\n\nYêu cầu: Trích dẫn chính xác điều khoản, lập luận logic và chặt chẽ."
USER_PROMPT = """{question}"""

In [ ]:
def generate_train_instruction(system_prompt, user_prompt, question, thinking_content, answer):
    instruction = f"<|im_start|>system\n{system_prompt}<|im_end|>\n<|im_start|>user\n{user_prompt.format(question=question)}<|im_end|>\n<|im_start|>assistant\n<think>\n{thinking_content}\n</think>\n\n{answer}<|im_end|>"
    return instruction

def generate_test_instruction(system_prompt, user_prompt, question):
    instruction = f"<|im_start|>system\n{system_prompt}<|im_end|>\n<|im_start|>user\n{user_prompt.format(question=question)}<|im_end|>\n<|im_start|>assistant\n"
    return instruction

In [ ]:
for index, values in tqdm(df_train.iterrows(), total=len(df_train), desc="Generating intruction prompt for training..."):
    QUESTION = df_train['question'][index]
    THINKING_CONTENT = df_train['thinking_content'][index]
    ANSWER = df_train['answer'][index]

    df_train.at[index, "instruction"] = generate_train_instruction(
        system_prompt=SYSTEM_PROMPT,
        user_prompt=USER_PROMPT,
        question=QUESTION,
        thinking_content=THINKING_CONTENT,
        answer=ANSWER
    )

for index, values in tqdm(df_val.iterrows(), total=len(df_val), desc="Generating intruction prompt for validation..."):
    QUESTION = df_val['question'][index]
    THINKING_CONTENT = df_val['thinking_content'][index]
    ANSWER = df_val['answer'][index]

    df_val.at[index, "instruction"] = generate_train_instruction(
        system_prompt=SYSTEM_PROMPT,
        user_prompt=USER_PROMPT,
        question=QUESTION,
        thinking_content=THINKING_CONTENT,
        answer=ANSWER
    )

for index, values in tqdm(df_test.iterrows(), total=len(df_test), desc="Generating intruction prompt for testing..."):
    QUESTION = df_test['question'][index]
    df_test.at[index, "instruction"] = generate_test_instruction(
        system_prompt=SYSTEM_PROMPT,
        user_prompt=USER_PROMPT,
        question=QUESTION,
    )

In [ ]:
print(df_train.iloc[0]['instruction'])

In [ ]:
df_train["len_instruction"] = df_train["instruction"].apply(lambda x: len(x.split(" ")))
df_val["len_instruction"] = df_val["instruction"].apply(lambda x: len(x.split(" ")))
df_test["len_instruction"] = df_test["instruction"].apply(lambda x: len(x.split(" ")))

In [ ]:
df_train["len_instruction"].max(), df_val["len_instruction"].max(), df_test["len_instruction"].max()

In [ ]:
dataset_train = Dataset.from_pandas(df_train, preserve_index=False)
dataset_val = Dataset.from_pandas(df_val, preserve_index=False)
dataset_test = Dataset.from_pandas(df_test, preserve_index=False)

In [ ]:
def convert_to_text_format(dataset):
    def map_func(examples):
        return {"text": examples["instruction"]}
    
    return dataset.map(map_func, batched=True, remove_columns=dataset.column_names)

dataset_train_formatted = convert_to_text_format(dataset_train)
dataset_val_formatted = convert_to_text_format(dataset_val)
dataset_test_formatted = convert_to_text_format(dataset_test)

# Simple formatting function
def formatting_func(examples):
    return {"text": examples["text"]}

In [ ]:
from trl import SFTTrainer
from transformers import TrainingArguments, DataCollatorForSeq2Seq

trainer = SFTTrainer(
    model=model,
    tokenizer=tokenizer,
    train_dataset=dataset_train_formatted,
    eval_dataset=dataset_val_formatted,
    dataset_text_field="text",
    max_seq_length=4096,
    data_collator=DataCollatorForSeq2Seq(tokenizer=tokenizer),
    dataset_num_proc=2,
    packing=False, # Can make training 5x faster for short sequences.
    formatting_func=formatting_func,
    args=TrainingArguments(
        per_device_train_batch_size=1,
        gradient_accumulation_steps=1, #4
        num_train_epochs=1,
        learning_rate=4.5e-4,
        fp16=True,
        bf16=False,
        logging_steps=300,
        eval_strategy="steps",
        eval_steps=300, 
        optim="paged_adamw_32bit",
        weight_decay=0.01,
        seed=3407,
        output_dir="outputs",
        report_to="none",
        save_strategy="no",
        save_total_limit=1
    ),
)

In [ ]:
trainer_stats = trainer.train()

# Inference

In [ ]:
class EndOfConversationCriteria(StoppingCriteria):
    def __init__(self, tokenizer):
        self.tokenizer = tokenizer
        self.end_token_id = tokenizer.encode("<|im_end|>", add_special_tokens=False)[0]
    
    def __call__(self, input_ids, scores, **kwargs):
        return input_ids[0][-1] == self.end_token_id

FastLanguageModel.for_inference(model)

stopping_criteria = StoppingCriteriaList([EndOfConversationCriteria(tokenizer)])

In [ ]:
df_test["generated_answer"] = ""

for index, values in tqdm(df_test.iterrows(), total=len(df_test), desc="Generating answer for test set..."):
    instruction = values["instruction"]
    gold_answer = values["answer"]

    inputs = tokenizer(
        instruction,
        return_tensors='pt',
        truncation=True,
        max_length=4096
    ).to('cuda')

    outputs = model.generate(
        input_ids=inputs.input_ids,
        attention_mask=inputs.attention_mask,
        max_new_tokens=2048,
        stopping_criteria=stopping_criteria,
        use_cache=True
    )

    generated_answer = tokenizer.batch_decode(outputs)[0].split("<|im_start|>assistant\n")[1].replace("<|endoftext|>", "")

    df_test.at[index, "generated_answer"] = generated_answer
    
    print(f'==================== Generate output: ====================\n{generated_answer}')
    print(f"==================== Ground truth:====================\n{values['answer']}\n")

In [ ]:
df_test

In [ ]:
df_test = df_test[["answer", "generated_answer"]]
df_test["generated_answer"] = df_test["generated_answer"].apply(lambda x:x.replace("<|im_end|>", ""))
df_test

In [ ]:
df_test.to_csv(r"Syllogism-reasoning.csv", index=False, encoding="utf-8-sig")